# 03a – Unsupervised Topic Modeling

**Methods:** LDA · NMF · (optional) BERTopic  
**Outputs:** `lda_topics.json` · `nmf_topics.json` · `topic_term_distributions.csv`


## 0. Setup

In [1]:
import os, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # headless rendering
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.preprocessing import normalize

import re

# ── Paths ─────────────────────────────────────────────────────────────────
REPO_ROOT   = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
RESULTS_DIR = os.path.join(REPO_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Hyper-parameters (tune here) ──────────────────────────────────────────
N_TOPICS       = 5     # number of topics to discover
N_TOP_WORDS    = 10    # top keywords per topic to display / save
MAX_ITER_LDA   = 50
MAX_ITER_NMF   = 200
RANDOM_STATE   = 42
print('Setup complete.')


Setup complete.


## 1. Load Data

In [2]:
# ── Load virtual dataset (generated by axis_2_topic_labeling.ipynb) ──────
DATA_PATH = os.path.join(RESULTS_DIR, 'virtual_tickets.csv')

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Dataset not found at {DATA_PATH}.\n'
        'Please run axis_2_topic_labeling.ipynb first to generate virtual_tickets.csv.'
    )

df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df)} tickets.')
df.head(3)


Loaded 200 tickets.


,ticket_id,text,true_topic,true_issue_type,true_sentiment,created_at
0,TKT-0001,I cannot log in to my account. The password re...,login,Account,negative,2024-05-20
1,TKT-0002,Error 500 is displayed whenever I submit the c...,bug,Bug,negative,2024-03-12
2,TKT-0003,Single sign-on with Google stopped working thi...,login,Account,negative,2024-10-06


## 2. Text Preprocessing

In [3]:
import re

STOP_WORDS = (
    'english'  # sklearn built-in; replace with a custom list if needed
)

def preprocess(text: str) -> str:
    """Lowercase, strip punctuation/numbers, collapse whitespace."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(preprocess)
print('Sample cleaned texts:')
for t in df['clean_text'].head(3):
    print(' •', t)


Sample cleaned texts:
 • i cannot log in to my account the password reset email never arrived
 • error is displayed whenever i submit the checkout form
 • single sign on with google stopped working this morning


## 3. Vectorisation

In [4]:
# ── Bag-of-Words for LDA ──────────────────────────────────────────────────
count_vec = CountVectorizer(
    max_df=0.95,
    min_df=2,
    max_features=500,
    stop_words=STOP_WORDS,
)
X_count = count_vec.fit_transform(df['clean_text'])
count_vocab = count_vec.get_feature_names_out()

# ── TF-IDF for NMF ────────────────────────────────────────────────────────
tfidf_vec = TfidfVectorizer(
    max_df=0.95,
    min_df=2,
    max_features=500,
    stop_words=STOP_WORDS,
)
X_tfidf = tfidf_vec.fit_transform(df['clean_text'])
tfidf_vocab = tfidf_vec.get_feature_names_out()

print(f'BoW  matrix: {X_count.shape}')
print(f'TF-IDF matrix: {X_tfidf.shape}')


BoW  matrix: (200, 166)
TF-IDF matrix: (200, 166)


## 4. LDA (Latent Dirichlet Allocation)

In [5]:
lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    max_iter=MAX_ITER_LDA,
    learning_method='batch',
    random_state=RANDOM_STATE,
)
lda.fit(X_count)
print(f'LDA perplexity: {lda.perplexity(X_count):.2f}')


LDA perplexity: 110.52


In [6]:
def top_words(model, vocab, n=N_TOP_WORDS):
    """Return list of (topic_id, [top-n words]) for every topic."""
    results = []
    for topic_idx, topic_vec in enumerate(model.components_):
        top_idx  = topic_vec.argsort()[::-1][:n]
        keywords = [vocab[i] for i in top_idx]
        results.append({'topic': topic_idx, 'keywords': keywords})
    return results

lda_topics = top_words(lda, count_vocab)
for t in lda_topics:
    print(f"Topic {t['topic']:2d}: {', '.join(t['keywords'])}")


Topic  0: need, data, email, account, mode, update, associated, address, records, blank
Topic  1: account, login, error, keeps, export, arrived, log, downloads, csv, page
Topic  2: app, workflow, improve, mobile, ios, significantly, button, change, does, display
Topic  3: working, single, sign, stopped, google, morning, subscription, applied, portal, correctly
Topic  4: invoice, month, billing, missing, history, need, refund, purchase, week, accidental


In [7]:
# ── Assign dominant topic to each ticket ──────────────────────────────────
lda_doc_topic = lda.transform(X_count)
df['lda_topic'] = lda_doc_topic.argmax(axis=1)
df['lda_topic_score'] = lda_doc_topic.max(axis=1)
print('LDA topic assignment sample:')
print(df[['ticket_id', 'text', 'lda_topic', 'lda_topic_score']].head(5))


LDA topic assignment sample:
  ticket_id                                               text  lda_topic  \
0  TKT-0001  I cannot log in to my account. The password re...          1   
1  TKT-0002  Error 500 is displayed whenever I submit the c...          1   
2  TKT-0003  Single sign-on with Google stopped working thi...          3   
3  TKT-0004  Forgot my password and the reset link expired ...          4   
4  TKT-0005  I cannot log in to my account. The password re...          1   

   lda_topic_score  
0         0.885081  
1         0.866384  
2         0.885642  
3         0.885506  
4         0.885081  


## 5. NMF (Non-Negative Matrix Factorisation)

In [8]:
nmf = NMF(
    n_components=N_TOPICS,
    max_iter=MAX_ITER_NMF,
    random_state=RANDOM_STATE,
    init='nndsvda',
)
nmf.fit(X_tfidf)

nmf_topics = top_words(nmf, tfidf_vocab)
for t in nmf_topics:
    print(f"Topic {t['topic']:2d}: {', '.join(t['keywords'])}")


Topic  0: reset, password, account, email, log, arrived, forgot, link, expired, use
Topic  1: button, downloads, csv, export, file, save, display, does, change, directly
Topic  2: billing, month, missing, history, invoice, applied, eu, portal, vat, correctly
Topic  3: workflow, significantly, improve, mobile, ios, app, try, crashes, larger, mb
Topic  4: need, week, accidental, purchase, refund, secondary, organisation, admin, add, user


In [9]:
# ── Assign dominant topic ─────────────────────────────────────────────────
nmf_doc_topic = nmf.transform(X_tfidf)
df['nmf_topic'] = nmf_doc_topic.argmax(axis=1)
df['nmf_topic_score'] = nmf_doc_topic.max(axis=1)
print('NMF topic assignment sample:')
print(df[['ticket_id', 'nmf_topic', 'nmf_topic_score']].head(5))


NMF topic assignment sample:
  ticket_id  nmf_topic  nmf_topic_score
0  TKT-0001          0         0.430647
1  TKT-0002          0         0.000257
2  TKT-0003          2         0.002294
3  TKT-0004          0         0.267449
4  TKT-0005          0         0.430647


## 6. (Optional) Determine Optimal Number of Topics

Iterate over a range of *k* values and track LDA perplexity and approximate coherence proxy (held-out log-likelihood).

In [10]:
# ── Perplexity vs k ───────────────────────────────────────────────────────
k_range = range(2, 9)
perplexities = []

for k in k_range:
    m = LatentDirichletAllocation(
        n_components=k, max_iter=30,
        learning_method='batch', random_state=RANDOM_STATE
    )
    m.fit(X_count)
    perplexities.append(m.perplexity(X_count))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(k_range), perplexities, marker='o', color='steelblue')
ax.set_xlabel('Number of Topics (k)')
ax.set_ylabel('Perplexity (lower = better)')
ax.set_title('LDA Perplexity vs Number of Topics')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'lda_perplexity_plot.png'), dpi=100)
plt.close()
print('Perplexity plot saved.')


Perplexity plot saved.


## 7. Visualisation – Topic-Term Heatmap

In [11]:
def topic_term_matrix(model, vocab, n=15):
    """Return a DataFrame of top-n term weights per topic."""
    comp = model.components_
    comp_norm = comp / comp.sum(axis=1, keepdims=True)
    top_global = np.argsort(comp_norm.sum(axis=0))[::-1][:n]
    terms = [vocab[i] for i in top_global]
    return pd.DataFrame(
        comp_norm[:, top_global],
        index=[f'Topic {i}' for i in range(comp_norm.shape[0])],
        columns=terms,
    )

for model_name, model, vocab in [
    ('LDA', lda, count_vocab),
    ('NMF', nmf, tfidf_vocab),
]:
    ttm = topic_term_matrix(model, vocab)
    fig, ax = plt.subplots(figsize=(14, 4))
    sns.heatmap(ttm, cmap='YlOrRd', ax=ax, linewidths=0.3)
    ax.set_title(f'{model_name} – Topic × Term Weight Heatmap')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'{model_name.lower()}_topic_term_heatmap.png'), dpi=100)
    plt.close()
    print(f'{model_name} heatmap saved.')


LDA heatmap saved.


NMF heatmap saved.


## 8. Save Outputs

In [12]:
# ── lda_topics.json ───────────────────────────────────────────────────────
lda_out_path = os.path.join(RESULTS_DIR, 'lda_topics.json')
with open(lda_out_path, 'w') as f:
    json.dump(lda_topics, f, indent=2)
print(f'Saved: {lda_out_path}')

# ── nmf_topics.json ───────────────────────────────────────────────────────
nmf_out_path = os.path.join(RESULTS_DIR, 'nmf_topics.json')
with open(nmf_out_path, 'w') as f:
    json.dump(nmf_topics, f, indent=2)
print(f'Saved: {nmf_out_path}')

# ── topic_term_distributions.csv ──────────────────────────────────────────
lda_ttm = topic_term_matrix(lda, count_vocab, n=N_TOP_WORDS)
nmf_ttm = topic_term_matrix(nmf, tfidf_vocab, n=N_TOP_WORDS)
lda_ttm.index = [f'LDA_Topic_{i}' for i in range(N_TOPICS)]
nmf_ttm.index = [f'NMF_Topic_{i}' for i in range(N_TOPICS)]
combined_ttm = pd.concat([lda_ttm, nmf_ttm])
ttm_path = os.path.join(RESULTS_DIR, 'topic_term_distributions.csv')
combined_ttm.to_csv(ttm_path)
print(f'Saved: {ttm_path}')

# ── ticket-level topic assignments ────────────────────────────────────────
topic_assign_path = os.path.join(RESULTS_DIR, 'ticket_topic_assignments.csv')
df[['ticket_id', 'true_topic', 'lda_topic', 'lda_topic_score',
    'nmf_topic', 'nmf_topic_score']].to_csv(topic_assign_path, index=False)
print(f'Saved: {topic_assign_path}')

print('\n03a complete.')


Saved: /home/runner/work/SDPA_EMATM0048/SDPA_EMATM0048/results/lda_topics.json
Saved: /home/runner/work/SDPA_EMATM0048/SDPA_EMATM0048/results/nmf_topics.json
Saved: /home/runner/work/SDPA_EMATM0048/SDPA_EMATM0048/results/topic_term_distributions.csv
Saved: /home/runner/work/SDPA_EMATM0048/SDPA_EMATM0048/results/ticket_topic_assignments.csv

03a complete.
